# Proof of Concept -- Local Search: N-Queens Problem

Notebook ini menguji seluruh algoritma local search yang diimplementasikan di `src/nqueens.py`:
1. **Hill-Climbing Basic** (Steepest-Ascent)
2. **Hill-Climbing dengan Sideways Move**
3. **Stochastic Hill-Climbing**
4. **Random Restart Hill-Climbing**
5. **Simulated Annealing**
6. **Genetic Algorithm**

Setiap percobaan menampilkan:
- State awal (random) dan state akhir
- Nilai *objective function* $h(S)$ selama proses pencarian
- Visualisasi papan N-Queens dalam bentuk teks

## Setup: Import modul dari `src`

In [ ]:
import sys
import os

# Tambahkan root project ke sys.path agar bisa import dari src
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.nqueens import (
    generate_random_state,
    calculate_h,
    get_neighbors,
    hill_climbing_basic,
    hill_climbing_sideways,
    hill_climbing_stochastic,
    hill_climbing_random_restart,
    simulated_annealing,
    genetic_algorithm,
    ga_fitness,
    print_board,
    print_state_array,
    print_separator,
    print_h_history,
    visualize_search,
)

print("Import berhasil!")

## Konfigurasi

In [ ]:
# Ukuran papan
N = 8
print(f"Ukuran papan: {N} x {N}")
print(f"Jumlah pasangan maksimum: {N*(N-1)//2}")

---
## 1. Basic Hill-Climbing (Steepest-Ascent)

Pada setiap iterasi, evaluasi **seluruh** neighbor dan pilih yang memiliki $h$ terkecil. Berhenti jika tidak ada neighbor yang lebih baik (local minimum).

In [ ]:
print_separator()
print("  ALGORITMA: BASIC HILL-CLIMBING (STEEPEST-ASCENT)")
print_separator()

state, h, h_history, states_history = hill_climbing_basic(N)

# State awal
print("\nSTATE AWAL:")
print_state_array(states_history[0], label="  Array")
print_board(states_history[0], h_val=h_history[0])

# Visualisasi proses pencarian
visualize_search(states_history, h_history, "Basic Hill-Climbing")

# State akhir
print("STATE AKHIR:")
print_state_array(state, label="  Array")
print_board(state, h_val=h)

# Status
if h == 0:
    print(f"[OK] SOLUSI DITEMUKAN setelah {len(h_history) - 1} iterasi!")
else:
    print(f"[X] TERJEBAK di local minimum (h = {h}) setelah {len(h_history) - 1} iterasi.")

# Riwayat h
print_h_history(h_history)

---
## 2. Hill-Climbing dengan Sideways Move

Mengizinkan perpindahan ke neighbor dengan $h$ sama (*sideways move*), dibatasi maksimal `max_sideways` langkah berturut-turut untuk menghindari infinite loop pada plateau.

In [ ]:
MAX_SIDEWAYS = 100

print_separator()
print("  ALGORITMA: HILL-CLIMBING DENGAN SIDEWAYS MOVE")
print_separator()
print(f"  Max sideways: {MAX_SIDEWAYS}")

state, h, h_history, states_history = hill_climbing_sideways(N, MAX_SIDEWAYS)

# State awal
print("\nSTATE AWAL:")
print_state_array(states_history[0], label="  Array")
print_board(states_history[0], h_val=h_history[0])

# Visualisasi proses pencarian
visualize_search(states_history, h_history, "HC Sideways Move")

# State akhir
print("STATE AKHIR:")
print_state_array(state, label="  Array")
print_board(state, h_val=h)

# Status
if h == 0:
    print(f"[OK] SOLUSI DITEMUKAN setelah {len(h_history) - 1} iterasi!")
else:
    print(f"[X] TERJEBAK (h = {h}) setelah {len(h_history) - 1} iterasi.")

print_h_history(h_history)

---
## 3. Stochastic Hill-Climbing

Alih-alih memilih neighbor terbaik, pilih secara **acak** di antara neighbor yang memberikan perbaikan ($h$ lebih kecil).

In [ ]:
print_separator()
print("  ALGORITMA: STOCHASTIC HILL-CLIMBING")
print_separator()

state, h, h_history, states_history = hill_climbing_stochastic(N)

# State awal
print("\nSTATE AWAL:")
print_state_array(states_history[0], label="  Array")
print_board(states_history[0], h_val=h_history[0])

# Visualisasi proses pencarian
visualize_search(states_history, h_history, "Stochastic Hill-Climbing")

# State akhir
print("STATE AKHIR:")
print_state_array(state, label="  Array")
print_board(state, h_val=h)

# Status
if h == 0:
    print(f"[OK] SOLUSI DITEMUKAN setelah {len(h_history) - 1} iterasi!")
else:
    print(f"[X] TERJEBAK di local minimum (h = {h}) setelah {len(h_history) - 1} iterasi.")

print_h_history(h_history)

---
## 4. Random Restart Hill-Climbing

Menjalankan basic hill-climbing berkali-kali dari initial state random baru. Jika terjebak di local minimum, restart dari konfigurasi baru.

In [ ]:
MAX_RESTARTS = 100

print_separator()
print("  ALGORITMA: RANDOM RESTART HILL-CLIMBING")
print_separator()
print(f"  Max restarts: {MAX_RESTARTS}")

(state, h, h_history, states_history,
 total_restarts, restart_points) = hill_climbing_random_restart(N, MAX_RESTARTS)

# State awal (restart pertama)
print("\nSTATE AWAL (Restart ke-1):")
print_state_array(states_history[0], label="  Array")
print_board(states_history[0], h_val=h_history[0])

# Info restart
print(f"Jumlah restart: {total_restarts}")
for i, rp in enumerate(restart_points):
    if rp < len(states_history):
        print(f"  Restart ke-{i+1}: h awal = {h_history[rp]}")

# State akhir
print("\nSTATE AKHIR:")
print_state_array(state, label="  Array")
print_board(state, h_val=h)

# Status
if h == 0:
    print(f"[OK] SOLUSI DITEMUKAN setelah {total_restarts} restart, "
          f"total {len(h_history) - 1} iterasi!")
else:
    print(f"[X] GAGAL setelah {MAX_RESTARTS} restart.")

# Riwayat h (ringkas)
if len(h_history) <= 30:
    print_h_history(h_history)
else:
    print(f"\nRiwayat h (total {len(h_history)} entri, ditampilkan awal & akhir):")
    for i in range(min(5, len(h_history))):
        print(f"  Iterasi {i:>4}: h = {h_history[i]}")
    print(f"  ...")
    for i in range(max(0, len(h_history) - 5), len(h_history)):
        print(f"  Iterasi {i:>4}: h = {h_history[i]}")

---
## 5. Simulated Annealing

Memperbolehkan perpindahan ke state yang **lebih buruk** dengan probabilitas $P = e^{-\Delta E / T}$. Suhu $T$ menurun secara eksponensial (*cooling schedule*).

In [ ]:
T0 = 100.0
ALPHA = 0.995
T_MIN = 0.01

print_separator()
print("  ALGORITMA: SIMULATED ANNEALING")
print_separator()
print(f"  T0 = {T0}, alpha = {ALPHA}, T_min = {T_MIN}")

(state, h, h_history, states_history, temp_history) = simulated_annealing(
    N, T0, ALPHA, T_MIN)

total_iter = len(h_history) - 1

# State awal
print("\nSTATE AWAL:")
print_state_array(states_history[0], label="  Array")
print_board(states_history[0], h_val=h_history[0])
print(f"  Suhu awal: T = {temp_history[0]:.2f}")

# Snapshot proses pencarian
print(f"\nProses pencarian ({total_iter} iterasi):")
print(f"{'Iterasi':>10} | {'h(S)':>6} | {'Suhu (T)':>12}")
print("-" * 35)
step = max(1, total_iter // 10)
for i in range(0, total_iter + 1, step):
    if i < len(h_history):
        print(f"{i:>10} | {h_history[i]:>6} | {temp_history[i]:>12.4f}")
if total_iter % step != 0:
    print(f"{total_iter:>10} | {h_history[-1]:>6} | {temp_history[-1]:>12.4f}")

# Visualisasi proses
visualize_search(states_history, h_history, "Simulated Annealing")

# State akhir
print("STATE AKHIR:")
print_state_array(state, label="  Array")
print_board(state, h_val=h)
print(f"  Suhu akhir: T = {temp_history[-1]:.6f}")

if h == 0:
    print(f"[OK] SOLUSI DITEMUKAN setelah {total_iter} iterasi!")
else:
    print(f"[X] GAGAL (h = {h}) setelah {total_iter} iterasi. Suhu telah mencapai minimum.")

---
## 6. Genetic Algorithm

Memelihara **populasi** individu yang berevolusi melalui *selection* (roulette wheel), *single-point crossover*, dan *mutation*.

In [ ]:
POP_SIZE = 100
MUTATION_RATE = 0.1
MAX_GEN = 500

print_separator()
print("  ALGORITMA: GENETIC ALGORITHM")
print_separator()
print(f"  Populasi: {POP_SIZE}, Mutation rate: {MUTATION_RATE}, Max generasi: {MAX_GEN}")

h_max = N * (N - 1) // 2

(best_ind, h, best_fit_hist, avg_fit_hist,
 gen_found, best_ind_history) = genetic_algorithm(
    N, POP_SIZE, MUTATION_RATE, MAX_GEN)

# Individu terbaik generasi pertama
print("\nINDIVIDU TERBAIK GENERASI 1:")
print_state_array(best_ind_history[0], label="  Array")
print_board(best_ind_history[0], h_val=h_max - best_fit_hist[0])
print(f"  Fitness: {best_fit_hist[0]} / {h_max}")

# Evolusi fitness
total_gen = len(best_fit_hist)
print(f"\nEvolusi fitness ({total_gen} generasi):")
print(f"{'Generasi':>10} | {'Best Fit':>10} | {'Avg Fit':>10} | {'Best h':>8}")
print("-" * 48)
step = max(1, total_gen // 10)
for i in range(0, total_gen, step):
    print(f"{i + 1:>10} | {best_fit_hist[i]:>10} | "
          f"{avg_fit_hist[i]:>10.2f} | {h_max - best_fit_hist[i]:>8}")
if (total_gen - 1) % step != 0:
    i = total_gen - 1
    print(f"{i + 1:>10} | {best_fit_hist[i]:>10} | "
          f"{avg_fit_hist[i]:>10.2f} | {h_max - best_fit_hist[i]:>8}")

# Visualisasi beberapa generasi
if len(best_ind_history) > 1:
    gen_indices = [0]
    step_vis = max(1, len(best_ind_history) // 4)
    for i in range(step_vis, len(best_ind_history) - 1, step_vis):
        gen_indices.append(i)
    gen_indices.append(len(best_ind_history) - 1)
    gen_indices = sorted(set(gen_indices))

    print(f"\nVisualisasi individu terbaik pada beberapa generasi:\n")
    for gi in gen_indices:
        gen_h = h_max - best_fit_hist[gi]
        print_board(best_ind_history[gi], label=f"Generasi {gi + 1}", h_val=gen_h)

# State akhir
print("INDIVIDU TERBAIK (STATE AKHIR):")
print_state_array(best_ind, label="  Array")
print_board(best_ind, h_val=h)

if h == 0:
    print(f"[OK] SOLUSI DITEMUKAN pada generasi {gen_found}!")
else:
    print(f"[X] GAGAL (h = {h}) setelah {MAX_GEN} generasi.")